In [2]:
import sys
import os
# 将项目根目录加入路径，以便识别 01_Environment 模块
sys.path.append(os.path.abspath('..'))

from stable_baselines3 import PPO
from stable_baselines3.common.env_checker import check_env
from stable_baselines3.common.callbacks import EvalCallback
from stable_baselines3.common.monitor import Monitor

from Environment.liquidation_env import LiquidationEnv

# 初始化环境
env = LiquidationEnv()

# 使用 SB3 自带工具检查环境是否符合规范 (如果不报错就说明环境写得很完美)
check_env(env)
print("环境检查通过！")


2026-04-08 11:02:16.193066: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-04-08 11:02:16.371344: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1775617336.453499   17768 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1775617336.486379   17768 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-04-08 11:02:16.701540: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instr

环境检查通过！


/home/koro/anaconda3/envs/finrl/lib/python3.9/site-packages/gymnasium/spaces/box.py:235: UserWarning: WARN: Box low's precision lowered by casting to float32, current low.dtype=float64
  gym.logger.warn(
/home/koro/anaconda3/envs/finrl/lib/python3.9/site-packages/gymnasium/spaces/box.py:305: UserWarning: WARN: Box high's precision lowered by casting to float32, current high.dtype=float64
  gym.logger.warn(


In [ ]:
# 使用 Monitor 包装环境以记录训练数据
log_dir = "../models/logs/"
os.makedirs(log_dir, exist_ok=True)
env = Monitor(LiquidationEnv(), log_dir)

# 评估环境与 Callback (边训练边测试)
eval_env = Monitor(LiquidationEnv(), log_dir)
eval_callback = EvalCallback(eval_env, best_model_save_path='../models/best_ppo_model/',
                             log_path=log_dir, eval_freq=2000,
                             deterministic=True, render=False)

# 初始化 PPO 模型
# MlpPolicy 适用于一维向量状态
# 在 train_ppo.ipynb 中修改
model = PPO(
    "MlpPolicy", 
    env, 
    verbose=1, 
    learning_rate=0.0007696558671593552,  # 更新这里
    n_steps=512,                          # 更新这里
    batch_size=256,                       # 更新这里
    gamma=0.9632994051086053,             # 更新这里
    # 其他原有参数...
)

# 开始训练 (由于是金融时序，建议步数设大一点，比如 100k 到 500k)
print("开始训练...")
model.learn(total_timesteps=100000, callback=eval_callback)

# 保存最终模型
model.save("../models/ppo_liquidation_final")
print("训练完成并保存模型！")


Using cuda device
Wrapping the env in a DummyVecEnv.


/home/koro/anaconda3/envs/finrl/lib/python3.9/site-packages/stable_baselines3/common/on_policy_algorithm.py:150: UserWarning: You are trying to run PPO on the GPU, but it is primarily intended to run on the CPU when not using a CNN policy (you are using ActorCriticPolicy which should be a MlpPolicy). See https://github.com/DLR-RM/stable-baselines3/issues/1245 for more info. You can pass `device='cpu'` or `export CUDA_VISIBLE_DEVICES=` to force using the CPU.Note: The model will train, but the GPU utilization will be poor and the training might take longer than on CPU.
  warnings.warn(


开始训练...
Eval num_timesteps=2000, episode_reward=-0.03 +/- 0.00
Episode length: 100.00 +/- 0.00
---------------------------------
| eval/              |          |
|    mean_ep_length  | 100      |
|    mean_reward     | -0.0328  |
| time/              |          |
|    total_timesteps | 2000     |
---------------------------------
New best mean reward!
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 100      |
|    ep_rew_mean     | -0.0491  |
| time/              |          |
|    fps             | 279      |
|    iterations      | 1        |
|    time_elapsed    | 7        |
|    total_timesteps | 2048     |
---------------------------------
Eval num_timesteps=4000, episode_reward=-0.03 +/- 0.00
Episode length: 100.00 +/- 0.00
------------------------------------------
| eval/                   |              |
|    mean_ep_length       | 100          |
|    mean_reward          | -0.0292      |
| time/                   |              |
|  

In [4]:
# 1. 批量训练不同 lambda 的模型
lambdas = [0.1, 0.5, 2.0]
for l in lambdas:
    print(f"正在为 lambda = {l} 训练模型...")
    # 创建特定参数的环境
    env = LiquidationEnv(lam=l, reward_scale=1e-3) 
    
    # 初始化并训练
    model = PPO("MlpPolicy", env, verbose=0, learning_rate=3e-4)
    model.learn(total_timesteps=50000) # 建议至少5万步以看到明显区分
    
    # 保存为不同的文件名
    model.save(f"../models/ppo_lam_{l}")

print("所有模型训练完成！")

正在为 lambda = 0.1 训练模型...


/home/koro/anaconda3/envs/finrl/lib/python3.9/site-packages/gymnasium/spaces/box.py:235: UserWarning: WARN: Box low's precision lowered by casting to float32, current low.dtype=float64
  gym.logger.warn(
/home/koro/anaconda3/envs/finrl/lib/python3.9/site-packages/gymnasium/spaces/box.py:305: UserWarning: WARN: Box high's precision lowered by casting to float32, current high.dtype=float64
  gym.logger.warn(


正在为 lambda = 0.5 训练模型...
正在为 lambda = 2.0 训练模型...
所有模型训练完成！


In [1]:
import os
import sys
from sb3_contrib import RecurrentPPO
from stable_baselines3.common.evaluation import evaluate_policy

# 确保能找到环境模块
sys.path.append(os.path.abspath('..'))
from Environment.liquidation_env import LiquidationEnv

print("初始化环境...")
env_lstm = LiquidationEnv(lam=0.5, reward_scale=1e-3)

# 1. 把 Optuna 跑出来的最优参数写在这里
optuna_params = {
    'learning_rate': 0.0007696558671593552,
    'n_steps': 512,
    'batch_size': 256,
    'gamma': 0.9632994051086053
}

# 2. 初始化 RecurrentPPO (带有 LSTM 的 PPO)
print("开始训练带有记忆 (LSTM) 的 PPO 模型...")
model_lstm = RecurrentPPO(
    "MlpLstmPolicy", 
    env_lstm, 
    verbose=1, 
    policy_kwargs=dict(lstm_hidden_size=64), # LSTM 隐藏层大小
    **optuna_params # 注入最优参数
)

# 3. 训练模型 (10万步左右)
model_lstm.learn(total_timesteps=100000)

# 4. 保存模型
model_lstm.save("../models/recurrent_ppo_optuna")
print("LSTM 模型训练并保存完毕！")

2026-04-09 02:14:27.813871: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-04-09 02:14:27.868979: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1775672067.896909   25535 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1775672067.903906   25535 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-04-09 02:14:27.972613: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instr

初始化环境...
开始训练带有记忆 (LSTM) 的 PPO 模型...


/home/koro/anaconda3/envs/finrl/lib/python3.9/site-packages/gymnasium/spaces/box.py:235: UserWarning: WARN: Box low's precision lowered by casting to float32, current low.dtype=float64
  gym.logger.warn(
/home/koro/anaconda3/envs/finrl/lib/python3.9/site-packages/gymnasium/spaces/box.py:305: UserWarning: WARN: Box high's precision lowered by casting to float32, current high.dtype=float64
  gym.logger.warn(


Using cuda device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.


/home/koro/anaconda3/envs/finrl/lib/python3.9/site-packages/stable_baselines3/common/utils.py:168: UserWarning: get_schedule_fn() is deprecated, please use FloatSchedule() instead
  warnings.warn("get_schedule_fn() is deprecated, please use FloatSchedule() instead")
/home/koro/anaconda3/envs/finrl/lib/python3.9/site-packages/stable_baselines3/common/utils.py:214: UserWarning: constant_fn() is deprecated, please use ConstantSchedule() instead
  warnings.warn("constant_fn() is deprecated, please use ConstantSchedule() instead")


---------------------------------
| rollout/           |          |
|    ep_len_mean     | 100      |
|    ep_rew_mean     | -0.0687  |
| time/              |          |
|    fps             | 143      |
|    iterations      | 1        |
|    time_elapsed    | 3        |
|    total_timesteps | 512      |
---------------------------------
-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 100         |
|    ep_rew_mean          | -0.0595     |
| time/                   |             |
|    fps                  | 87          |
|    iterations           | 2           |
|    time_elapsed         | 11          |
|    total_timesteps      | 1024        |
| train/                  |             |
|    approx_kl            | 0.013792995 |
|    clip_fraction        | 0.0896      |
|    clip_range           | 0.2         |
|    entropy_loss         | -1.42       |
|    explained_variance   | -1.78       |
|    learning_rate        | 0.

In [2]:
# ==========================================
# 验证永久冲击 (gamma2) 路径无关性的批量训练
# ==========================================
import os
from stable_baselines3 import PPO

# 你上传的 PDE 图中使用的 gamma2 值
gamma2_values = [0.0, 1.5, 3.0]

# 使用 Optuna 找到的最优参数以确保模型收敛到理论最优解
optuna_params = {
    'learning_rate': 0.0007696558671593552,
    'n_steps': 512,
    'batch_size': 256,
    'gamma': 0.9632994051086053
}

for g2 in gamma2_values:
    print(f"\n>>> 正在为 gamma2 = {g2} 训练专属模型...")
    
    # 初始化环境：保持其他参数默认，只改变 gamma2
    env_g2 = LiquidationEnv(gamma2=g2, reward_scale=1e-3) 
    
    # 初始化模型
    model_g2 = PPO("MlpPolicy", env_g2, verbose=0, **optuna_params)
    
    # 训练 50000 步足以验证轨迹形状
    model_g2.learn(total_timesteps=50000) 
    
    # 保存模型
    save_path = f"../models/ppo_gamma2_{g2}"
    model_g2.save(save_path)
    print(f"模型已保存至: {save_path}.zip")

print("\n所有 gamma2 专家模型训练完成！")


>>> 正在为 gamma2 = 0.0 训练专属模型...


/home/koro/anaconda3/envs/finrl/lib/python3.9/site-packages/gymnasium/spaces/box.py:235: UserWarning: WARN: Box low's precision lowered by casting to float32, current low.dtype=float64
  gym.logger.warn(
/home/koro/anaconda3/envs/finrl/lib/python3.9/site-packages/gymnasium/spaces/box.py:305: UserWarning: WARN: Box high's precision lowered by casting to float32, current high.dtype=float64
  gym.logger.warn(
/home/koro/anaconda3/envs/finrl/lib/python3.9/site-packages/stable_baselines3/common/on_policy_algorithm.py:150: UserWarning: You are trying to run PPO on the GPU, but it is primarily intended to run on the CPU when not using a CNN policy (you are using ActorCriticPolicy which should be a MlpPolicy). See https://github.com/DLR-RM/stable-baselines3/issues/1245 for more info. You can pass `device='cpu'` or `export CUDA_VISIBLE_DEVICES=` to force using the CPU.Note: The model will train, but the GPU utilization will be poor and the training might take longer than on CPU.
  warnings.warn

模型已保存至: ../models/ppo_gamma2_0.0.zip

>>> 正在为 gamma2 = 1.5 训练专属模型...
模型已保存至: ../models/ppo_gamma2_1.5.zip

>>> 正在为 gamma2 = 3.0 训练专属模型...
模型已保存至: ../models/ppo_gamma2_3.0.zip

所有 gamma2 专家模型训练完成！


In [1]:
import sys
import os
sys.path.append(os.path.abspath('..'))
from Environment.liquidation_env import LiquidationEnv
from stable_baselines3 import PPO

print("初始化终极对决环境 (严格对齐 PDE 参数)...")
# 这里的参数完全照抄他 run_parameter_experiments.py 里的 base_model
# 注意：N=80 必须加上，因为他的 Nt=80
env_ultimate = LiquidationEnv(
    T=1.0, q0=1.0, S0=100.0, sigma=0.20, lam=0.5, alpha=120.0,
    gamma1=2.0, gamma2=1.5, eta1=3.0, eta2=2.0,
    N=80, 
    reward_scale=1e-3
)

# 填入你用 Optuna 跑出来的最牛参数
optuna_params = {
    'learning_rate': 0.0007696558671593552,
    'n_steps': 512,
    'batch_size': 256,
    'gamma': 0.9632994051086053
}

print("开始训练 PPO 终极模型...")
model_ultimate = PPO("MlpPolicy", env_ultimate, verbose=1, **optuna_params)
# 因为冲击变大了，环境变得更严苛，我们跑 20 万步确保它彻底收敛
model_ultimate.learn(total_timesteps=200000)

model_ultimate.save("../models/ppo_ultimate_baseline")
print("终极模型已保存至: ppo_ultimate_baseline.zip")

2026-04-09 04:24:30.250709: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-04-09 04:24:30.333309: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1775679870.355132    1282 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1775679870.361869    1282 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-04-09 04:24:30.446197: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instr

初始化终极对决环境 (严格对齐 PDE 参数)...
开始训练 PPO 终极模型...


/home/koro/anaconda3/envs/finrl/lib/python3.9/site-packages/gymnasium/spaces/box.py:235: UserWarning: WARN: Box low's precision lowered by casting to float32, current low.dtype=float64
  gym.logger.warn(
/home/koro/anaconda3/envs/finrl/lib/python3.9/site-packages/gymnasium/spaces/box.py:305: UserWarning: WARN: Box high's precision lowered by casting to float32, current high.dtype=float64
  gym.logger.warn(


Using cuda device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.


/home/koro/anaconda3/envs/finrl/lib/python3.9/site-packages/stable_baselines3/common/on_policy_algorithm.py:150: UserWarning: You are trying to run PPO on the GPU, but it is primarily intended to run on the CPU when not using a CNN policy (you are using ActorCriticPolicy which should be a MlpPolicy). See https://github.com/DLR-RM/stable-baselines3/issues/1245 for more info. You can pass `device='cpu'` or `export CUDA_VISIBLE_DEVICES=` to force using the CPU.Note: The model will train, but the GPU utilization will be poor and the training might take longer than on CPU.
  warnings.warn(


---------------------------------
| rollout/           |          |
|    ep_len_mean     | 80       |
|    ep_rew_mean     | -0.0683  |
| time/              |          |
|    fps             | 334      |
|    iterations      | 1        |
|    time_elapsed    | 1        |
|    total_timesteps | 512      |
---------------------------------
------------------------------------------
| rollout/                |              |
|    ep_len_mean          | 80           |
|    ep_rew_mean          | -0.0568      |
| time/                   |              |
|    fps                  | 322          |
|    iterations           | 2            |
|    time_elapsed         | 3            |
|    total_timesteps      | 1024         |
| train/                  |              |
|    approx_kl            | 0.0044005318 |
|    clip_fraction        | 0.0279       |
|    clip_range           | 0.2          |
|    entropy_loss         | -1.42        |
|    explained_variance   | -2.02        |
|    learning_r